# Statistics Interview Preparation

This notebook demonstrates interview-relevant statistical concepts using Python. Run the cells in order. Install dependencies if needed: `pip install numpy pandas scipy matplotlib`.

## 1. Descriptive statistics

Compare mean and median and inspect spread. A few extreme values can pull the mean away from a typical observation.

In [ ]:
import numpy as np
import pandas as pd

values = [2, 4, 4, 4, 5, 5, 7, 9]
series = pd.Series(values)

print("mean:", series.mean())
print("median:", series.median())
print("sample variance:", series.var())  # ddof=1 by default
print("sample standard deviation:", series.std())
print("IQR:", series.quantile(0.75) - series.quantile(0.25))


## 2. Outliers and robust summaries

In [ ]:
data = pd.Series([10, 11, 12, 12, 13, 14, 100])
print("mean:", data.mean())
print("median:", data.median())
print("Q1:", data.quantile(0.25), "Q3:", data.quantile(0.75))
print("IQR:", data.quantile(0.75) - data.quantile(0.25))


## 3. Probability and Bayes' theorem

The positive predictive value depends on prevalence as well as test sensitivity and specificity.

In [ ]:
prevalence = 0.01
sensitivity = 0.90
specificity = 0.95

true_positive = prevalence * sensitivity
false_positive = (1 - prevalence) * (1 - specificity)
ppv = true_positive / (true_positive + false_positive)
print(f"P(condition | positive) = {ppv:.3%}")


## 4. Sampling variability and standard error

In [ ]:
rng = np.random.default_rng(42)
population = rng.normal(loc=50, scale=10, size=100_000)
sample_means = [
    rng.choice(population, size=100, replace=True).mean()
    for _ in range(1000)
]
print("mean of sample means:", np.mean(sample_means))
print("empirical SD of sample means:", np.std(sample_means, ddof=1))
print("theoretical approximate SE:", np.std(population, ddof=1) / np.sqrt(100))


## 5. Confidence interval for a mean

This example uses a t interval, which accounts for estimating the population standard deviation from the sample.

In [ ]:
from scipy import stats

sample = np.array([12, 13, 15, 11, 14, 16, 13, 12, 15, 14], dtype=float)
n = len(sample)
mean = sample.mean()
se = stats.sem(sample)
critical = stats.t.ppf(0.975, df=n - 1)
ci = (mean - critical * se, mean + critical * se)
print("mean:", mean, "SE:", se, "95% t interval:", ci)


## 6. Hypothesis testing: Welch's t-test

In [ ]:
control = np.array([4.1, 3.9, 4.0, 4.3, 3.8, 4.2, 4.1])
treatment = np.array([4.4, 4.2, 4.1, 4.5, 4.0, 4.6, 4.3])

result = stats.ttest_ind(treatment, control, equal_var=False)
print("t statistic:", result.statistic)
print("p-value:", result.pvalue)
print("mean difference:", treatment.mean() - control.mean())
# H0: the population means are equal.
# A p-value alone does not measure effect size or practical value.


## 7. Correlation

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([1, 2, 2, 4, 5, 8])

print("Pearson:", stats.pearsonr(x, y))
print("Spearman:", stats.spearmanr(x, y))


## 8. A/B test: absolute change and relative lift

In [ ]:
control_rate = 0.04
treatment_rate = 0.05
absolute_change = treatment_rate - control_rate
relative_lift = absolute_change / control_rate

print(f"Absolute change: {absolute_change:.1%} ({absolute_change * 100:.1f} percentage point)")
print(f"Relative lift: {relative_lift:.1%}")


## 9. Bootstrap confidence interval for the mean

In [ ]:
def bootstrap_mean_ci(values, repetitions=3000, confidence=0.95, seed=42):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        raise ValueError("values must not be empty")
    if repetitions <= 0 or not 0 < confidence < 1:
        raise ValueError("invalid repetitions or confidence")
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(repetitions, values.size), replace=True)
    means = samples.mean(axis=1)
    alpha = 1 - confidence
    return tuple(np.quantile(means, [alpha / 2, 1 - alpha / 2]))

print(bootstrap_mean_ci([2, 4, 4, 5, 7, 9]))


## 10. Binary classification metrics

In [ ]:
def binary_metrics(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    if len(y_true) != len(y_pred) or not y_true:
        raise ValueError("labels must have equal non-zero length")
    if any(v not in (0, 1) for v in y_true + y_pred):
        raise ValueError("labels must be binary")
    tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
    fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
    fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

print(binary_metrics([1, 0, 1, 1, 0], [1, 0, 0, 1, 1]))


## Interview prompts

1. Why can high accuracy be misleading with imbalanced classes?
2. Why does statistical significance not guarantee practical significance?
3. How would you choose between a paired and independent t-test?
4. What can go wrong if an A/B test is checked repeatedly and stopped when p < 0.05?
5. How would you communicate an interval estimate to a non-technical stakeholder?